# Verify simplex / cerebellar coordinates in injection-structured.json

This notebook reads the raw JSON and prints exactly what is stored. No filtering,
no interpretation baked in — you see the actual records and decide.

Set the path in Cell 1, then Run All.

In [ ]:
# Cell 1 — load the file
import json, os

PATH = 'react-app/public/data/injection-structured.json'   # <-- edit if needed

d = json.load(open(PATH, encoding='utf-8'))
print('file:', os.path.abspath(PATH))
print('size (MB):', round(os.path.getsize(PATH)/1024/1024, 2))
print('total papers:', len(d))
# how many targets total
nt = sum(len(r.get('targets', [])) for r in d.values() if isinstance(r, dict))
print('total targets:', nt)

In [ ]:
# Cell 2 — dump the exact record for PMID 31934857 (the disputed one)
pmid = '31934857'
rec = d.get(pmid)
if rec is None:
    print(pmid, 'NOT in file')
else:
    print(json.dumps(rec, indent=2, ensure_ascii=False))

In [ ]:
# Cell 3 — EVERY target whose region text mentions simplex / simple lobule
import re
rx = re.compile(r'simplex|simple\s*lobule|lobulus\s*simplex|\bSIM\b', re.I)

hits = []
for k, rec in d.items():
    if not isinstance(rec, dict):
        continue
    for t in rec.get('targets', []):
        if not isinstance(t, dict):
            continue
        blob = f"{t.get('ccf_region','')} {t.get('region_verbatim','')} {t.get('source_quote','')}"
        if rx.search(blob):
            hits.append((k, t))

print(f'{len(hits)} simplex-matching targets across {len(set(h[0] for h in hits))} papers\n')
for pmid, t in hits:
    print('='*80)
    print('PMID:', pmid)
    print('  ccf_region     :', t.get('ccf_region'))
    print('  region_verbatim:', t.get('region_verbatim'))
    print('  AP / ML / DV   :', t.get('ap_mm'), '/', t.get('ml_mm'), '/', t.get('dv_mm'))
    print('  reference      :', t.get('reference'))
    print('  volume_nl      :', t.get('volume_nl'))
    print('  rate_nl_min    :', t.get('rate_nl_min'))
    print('  source_quote   :', t.get('source_quote'))

In [ ]:
# Cell 4 — does each numeric value actually appear in its own source_quote?
# (catches fabricated / cross-contaminated numbers: a value with no textual support)
def in_quote(val, quote):
    if val is None or quote is None:
        return None
    q = str(quote).replace('\u2212', '-')
    # match the number ignoring trailing zeros (2.2 matches '2.2', '2.20')
    s = str(val)
    nums = re.findall(r'-?\d+\.?\d*', s)
    return all(n in q for n in nums) if nums else None

print('For each simplex target: is the value literally present in its source_quote?\n')
for pmid, t in hits:
    q = t.get('source_quote') or ''
    print(f"PMID {pmid} | {t.get('ccf_region')}")
    for field in ('ap_mm', 'ml_mm', 'dv_mm', 'volume_nl', 'rate_nl_min'):
        v = t.get(field)
        chk = in_quote(v, q)
        tag = {True: 'in quote', False: 'NOT IN QUOTE', None: '(null / n-a)'}[chk]
        print(f"    {field:12} = {str(v):20} -> {tag}")
    print('    quote:', q[:160])
    print()

In [ ]:
# Cell 5 — implausibly small volume / rate across the WHOLE file (the 0.025 class)
small = []
for k, rec in d.items():
    if not isinstance(rec, dict):
        continue
    for t in rec.get('targets', []):
        if not isinstance(t, dict):
            continue
        v, r = t.get('volume_nl'), t.get('rate_nl_min')
        if (isinstance(v, (int, float)) and 0 < v < 0.1) or (isinstance(r, (int, float)) and 0 < r < 0.1):
            small.append((k, t.get('ccf_region'), v, r, (t.get('source_quote') or '')[:100]))

print(f'targets with volume<0.1 nL OR rate<0.1 nL/min : {len(small)}\n')
for k, reg, v, r, q in small[:40]:
    print(f'  {k} | {reg} | vol={v} rate={r}')
    print(f'       quote: {q}')

In [ ]:
# Cell 6 — sanity distribution of all non-null volumes & rates (spot the outliers)
vols  = [t.get('volume_nl')  for r in d.values() if isinstance(r, dict)
         for t in r.get('targets', []) if isinstance(t, dict)
         and isinstance(t.get('volume_nl'), (int, float))]
rates = [t.get('rate_nl_min') for r in d.values() if isinstance(r, dict)
         for t in r.get('targets', []) if isinstance(t, dict)
         and isinstance(t.get('rate_nl_min'), (int, float))]

def summarize(name, xs):
    xs = sorted(xs)
    if not xs:
        print(name, ': none'); return
    n = len(xs)
    print(f'{name}: n={n}  min={xs[0]}  p05={xs[int(n*0.05)]}  median={xs[n//2]}  p95={xs[int(n*0.95)]}  max={xs[-1]}')
    print(f'    below 0.1 : {sum(1 for x in xs if x < 0.1)}')
    print(f'    0.1 - 1   : {sum(1 for x in xs if 0.1 <= x < 1)}')
    print(f'    1 - 50    : {sum(1 for x in xs if 1 <= x < 50)}')
    print(f'    >= 50     : {sum(1 for x in xs if x >= 50)}')

summarize('volume_nl ', vols)
print()
summarize('rate_nl_min', rates)